In [37]:
import os 
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from src.config import BDIFF_RAW_DIR, GEO_DATA_RAW_DIR

load_dotenv(override=True)

HOST = os.getenv("MYSQL_HOST")
PORT = int(os.getenv("MYSQL_PORT"))
USER = os.getenv("MYSQL_USER")
PASSWORD = os.getenv("MYSQL_PASSWORD")

engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}"
    f"?charset=utf8mb4"
)

# Test de connexion
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT VERSION()"))
        print("Connexion réussie !")
        print("Version MySQL :", result.scalar())
        
        # Lister les bases existantes
        dbs = conn.execute(text("SHOW DATABASES")).fetchall()
        print("Bases de données :", [db[0] for db in dbs])
except Exception as e:
    print("Erreur de connexion :", e)

Connexion réussie !
Version MySQL : 5.5.68-MariaDB
Bases de données : ['information_schema', 'arclight-igor', 'arclight-princeps', 'arclight-quad', 'incendies', 'mysql', 'performance_schema', 'prediction_incendies', 'test']


In [38]:
DATABASE_NAME = "incendies"

with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}"))
    conn.execute(text(f"USE {DATABASE_NAME}"))
    print(f"Base '{DATABASE_NAME}' créée ou déjà existante")

Base 'incendies' créée ou déjà existante


In [34]:
def get_engine():
    return create_engine(
        f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE_NAME}"
        f"?charset=utf8mb4"
    )

engine = get_engine()

## Table tmp_incendie

In [18]:
create_table_query = """
CREATE TABLE IF NOT EXISTS tmp_incendie (
    `Annee`                                     INT             NOT NULL,
    `Numero`                                    INT             NOT NULL,
    `Departement`                               VARCHAR(10)     NOT NULL,
    `Insee`                                     VARCHAR(10)     NOT NULL,
    `Nom_Commune`                               VARCHAR(150)    NULL,
    `Date_Premiere_Alerte`                      VARCHAR(30)     NOT NULL,
    `Surface_Parcourue`                         BIGINT          NOT NULL,
    `Surface_Foret`                             DECIMAL(15,2)   NULL,
    `Surface_Maquis_Garrigues`                  DECIMAL(15,2)   NULL,
    `Autres_Surfaces_Naturelles`                DECIMAL(15,2)   NULL,
    `Surfaces_Agricoles`                        DECIMAL(15,2)   NULL,
    `Autres_Surfaces`                           DECIMAL(15,2)   NULL,
    `Surface_Autres_Terres_Boisees`             DECIMAL(15,2)   NULL,
    `Surfaces_Non_Boisees_Naturelles`           DECIMAL(15,2)   NULL,
    `Surfaces_Non_Boisees_Artificialisees`      DECIMAL(15,2)   NULL,
    `Surfaces_Non_Boisees`                      DECIMAL(15,2)   NULL,
    `Precision_Surfaces`                        VARCHAR(100)    NULL,
    `Type_Peuplement`                           FLOAT           NULL,
    `Nature`                                    VARCHAR(100)    NULL,
    `Deces_BatimentsTouches`                    VARCHAR(100)    NULL,
    `Nombre_Deces`                              SMALLINT        NULL,
    `Nombre_Batiments_Totalement_Detruits`      SMALLINT        NULL,
    `Nombre_Batiments_Partiellement_Detruits`   SMALLINT        NULL,
    `Precision_Donnee`                          VARCHAR(200)    NULL
) ENGINE = MyISAM DEFAULT CHARSET = utf8mb4 COLLATE = utf8mb4_unicode_ci;
"""

with engine.connect() as conn:
    conn.execute(text(create_table_query))
    print("Table 'tmp_incendie' créée")

Table 'tmp_incendie' créée


In [19]:
indexes = [
    "CREATE INDEX idx_tmp_incendie_insee ON tmp_incendie (Insee);",
    "CREATE INDEX idx_tmp_incendie_nature ON tmp_incendie (Nature);",
    "CREATE INDEX idx_tmp_incendie_type_peuplement ON tmp_incendie (Type_Peuplement);",
    "CREATE INDEX idx_tmp_incendie_precision_surfaces ON tmp_incendie (Precision_Surfaces);",
]

with engine.connect() as conn:
    for sql in indexes:
        conn.execute(text(sql))
    conn.commit()

In [39]:
data_folder = BDIFF_RAW_DIR
fichiers_skiprows = {
    "Incendies2006.csv": 5,
    "Incendies2011.csv": 5,
    "Incendies2016.csv": 3,
    "Incendies2021.csv": 3,
}

all_dfs = []
for fichier, skiprows in fichiers_skiprows.items():
    file = os.path.join(data_folder, fichier)
    try:
        df = pd.read_csv(
            file,
            skiprows=skiprows,
            sep=';',
            # encoding='utf-8'
            encoding="utf-8-sig"
        )
        annee = fichier.replace("Incendies", "").replace(".csv", "")
        
        print(f"{fichier:<25} → skiprows={skiprows:2d} | Lignes={len(df):6,} | Colonnes={len(df.columns)}")
        
        all_dfs.append(df)
    except FileNotFoundError:
        print(f"Fichier non trouvé : {file}")
    except Exception as e:
        print(f"Erreur avec {fichier} : {e}")
print(f"\n {len(all_dfs)} fichiers chargés avec succès !")


Incendies2006.csv         → skiprows= 5 | Lignes=13,246 | Colonnes=24
Incendies2011.csv         → skiprows= 5 | Lignes=12,121 | Colonnes=24
Incendies2016.csv         → skiprows= 3 | Lignes=14,059 | Colonnes=24
Incendies2021.csv         → skiprows= 3 | Lignes=13,380 | Colonnes=24

 4 fichiers chargés avec succès !


In [26]:
def clean_column_names(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df.columns = df.columns.str.replace(r'\s+', ' ', regex=True)
    rename_dict = {
        "Année": "Annee",
        "Numéro": "Numero",
        "Département": "Departement",
        "Code INSEE": "Insee",
        "Nom de la commune": "Nom_Commune",
        "Date de première alerte": "Date_Premiere_Alerte",
        "Surface parcourue (m2)": "Surface_Parcourue",
        "Surface forêt (m2)": "Surface_Foret",
        "Surface maquis garrigues (m2)": "Surface_Maquis_Garrigues",
        "Autres surfaces naturelles hors forêt (m2)": "Autres_Surfaces_Naturelles",
        "Surfaces agricoles (m2)": "Surfaces_Agricoles",
        "Autres surfaces (m2)": "Autres_Surfaces",
        "Surface autres terres boisées (m2)": "Surface_Autres_Terres_Boisees",
        "Surfaces non boisées naturelles (m2)": "Surfaces_Non_Boisees_Naturelles",
        "Surfaces non boisées artificialisées (m2)": "Surfaces_Non_Boisees_Artificialisees",
        "Surfaces non boisées (m2)": "Surfaces_Non_Boisees",
        "Précision des surfaces": "Precision_Surfaces",
        "Type de peuplement": "Type_Peuplement",
        "Nature": "Nature",
        "Décès ou bâtiments touchés": "Deces_BatimentsTouches",
        "Nombre de décès": "Nombre_Deces",
        "Nombre de bâtiments totalement détruits": "Nombre_Batiments_Totalement_Detruits",
        "Nombre de bâtiments partiellement détruits": "Nombre_Batiments_Partiellement_Detruits",
        "Précision de la donnée": "Precision_Donnee"
    }
    df = df.rename(columns=rename_dict)
    return df

TABLE_NAME = "tmp_incendie"
total_inserted = 0
for i, df in enumerate(all_dfs):
    try:
        df_clean = clean_column_names(df)
        df_clean.to_sql(
                TABLE_NAME, 
                engine, 
                schema=DATABASE_NAME,
                if_exists="append", 
                index=False,
                chunksize=5000
            )
        rows = len(df_clean)
        total_inserted += rows
        print(f"{list(fichiers_skiprows.keys())[i]:<25} → {rows:6,} lignes insérées")
        
    except Exception as e:
        print(f"Erreur insertion {list(fichiers_skiprows.keys())[i]} : {e}")

print(f"\nFIN ! Total inséré dans tmp_incendie : {total_inserted:,} lignes")


Incendies2006.csv         → 13,246 lignes insérées
Incendies2011.csv         → 12,121 lignes insérées
Incendies2016.csv         → 14,059 lignes insérées
Incendies2021.csv         → 13,380 lignes insérées

FIN ! Total inséré dans tmp_incendie : 52,806 lignes


## Table tmp_commune

In [27]:
create_table = """
CREATE TABLE IF NOT EXISTS tmp_commune (
    `code_insee`                        VARCHAR(10)     NOT NULL,
    `nom_standard`                      VARCHAR(100)    NOT NULL,
    `nom_sans_pronom`                   VARCHAR(100)    NOT NULL,
    `nom_a`                             VARCHAR(100)    NOT NULL,
    `nom_de`                            VARCHAR(100)    NOT NULL,
    `nom_sans_accent`                   VARCHAR(100)    NOT NULL,
    `nom_standard_majuscule`            VARCHAR(100)    NOT NULL,
    `typecom`                           VARCHAR(10)     NOT NULL,
    `typecom_texte`                     VARCHAR(50)     NOT NULL,
    `reg_code`                          SMALLINT        NOT NULL,
    `reg_nom`                           VARCHAR(100)    NOT NULL,
    `dep_code`                          VARCHAR(10)     NOT NULL,
    `dep_nom`                           VARCHAR(100)    NOT NULL,
    `canton_code`                       VARCHAR(10)     NULL,
    `canton_nom`                        VARCHAR(100)    NULL,
    `epci_code`                         VARCHAR(20)     NULL,
    `epci_nom`                          VARCHAR(150)    NULL,
    `academie_code`                     SMALLINT        NOT NULL,
    `academie_nom`                      VARCHAR(100)    NOT NULL,
    `code_postal`                       VARCHAR(10)     NULL,
    `codes_postaux`                     VARCHAR(200)    NULL,
    `zone_emploi`                       INT             NULL,
    `code_insee_centre_zone_emploi`     VARCHAR(10)     NULL,
    `code_unite_urbaine`                VARCHAR(10)     NULL,
    `nom_unite_urbaine`                 VARCHAR(100)    NULL,
    `taille_unite_urbaine`              SMALLINT        NULL,
    `type_commune_unite_urbaine`        VARCHAR(50)     NULL,
    `statut_commune_unite_urbaine`      VARCHAR(50)     NULL,
    `population`                        INT             NOT NULL,
    `superficie_hectare`                INT             NOT NULL,
    `superficie_km2`                    INT             NOT NULL,
    `densite`                           DECIMAL(10,2)   NULL,
    `altitude_moyenne`                  SMALLINT        NOT NULL,
    `altitude_minimale`                 SMALLINT        NOT NULL,
    `altitude_maximale`                 SMALLINT        NOT NULL,
    `latitude_mairie`                   DECIMAL(9,6)    NOT NULL,
    `longitude_mairie`                  DECIMAL(9,6)    NOT NULL,
    `latitude_centre`                   DECIMAL(9,6)    NULL,
    `longitude_centre`                  DECIMAL(9,6)    NULL,
    `grille_densite`                    TINYINT         NOT NULL,
    `grille_densite_texte`              VARCHAR(50)     NOT NULL,
    `niveau_equipements_services`       TINYINT         NULL,
    `niveau_equipements_services_texte` VARCHAR(100)    NULL,
    `gentile`                           VARCHAR(100)    NULL,
    `url_wikipedia`                     VARCHAR(255)    NULL,
    `url_villedereve`                   VARCHAR(255)    NOT NULL
) ENGINE = MyISAM DEFAULT CHARSET = utf8mb4 COLLATE = utf8mb4_unicode_ci;
"""

with engine.connect() as conn:
    conn.execute(text(create_table))
    print("✅ Table 'tmp_commune' créée avec succès (MyISAM)")

✅ Table 'tmp_commune' créée avec succès (MyISAM)


In [28]:
indexes = [
    "CREATE INDEX idx_tmp_commune_code_insee ON tmp_commune (code_insee);",
    "CREATE INDEX idx_tmp_commune_longitude_mairie ON tmp_commune (longitude_mairie);",
    "CREATE INDEX idx_tmp_commune_latitude_mairie ON tmp_commune (latitude_mairie);",
]

with engine.connect() as conn:
    for sql in indexes:
        conn.execute(text(sql))
    conn.commit()

In [ ]:
data_folder = GEO_DATA_RAW_DIR
fichier = "communes-france-2025.csv"

input_file=os.path.join(data_folder, fichier)

df = pd.read_csv(input_file,
                sep=',',
                encoding='utf-8-sig')
df.head()

total_inserted = 0
try:
    df = df.drop(columns=["numero"])
    df.to_sql(
            'tmp_commune', 
            engine, 
            schema=DATABASE_NAME,
            if_exists="append", 
            index=False,
            chunksize=5000
        )
    rows = len(df)
    total_inserted += rows
    print(f"→ {rows:6,} lignes insérées")
    
except Exception as e:
    print(f"Erreur insertion : {e}")

print(f"\nFIN ! Total inséré dans tmp_commune : {total_inserted:,} lignes")

## Tables en lien avec la commune

In [13]:
# localisation
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE localisation (
            id_localisation INT AUTO_INCREMENT PRIMARY KEY,
            longitude decimal(9,6),
            latitude decimal(9,6)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
    """))

    conn.execute(text("""
        INSERT INTO localisation (longitude, latitude)
        SELECT
            distinct longitude_mairie, latitude_mairie
        FROM tmp_commune
    """))
    
    conn.commit()

In [14]:
indexes = [
    "CREATE INDEX idx_localisation_longitude ON localisation (longitude);",
    "CREATE INDEX idx_localisation_latitude ON localisation (latitude);",
]

with engine.connect() as conn:
    for sql in indexes:
        conn.execute(text(sql))
    conn.commit()

In [15]:
# region
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE region (
            id SMALLINT(6) AUTO_INCREMENT PRIMARY KEY,
            code SMALLINT(6),
            nom VARCHAR(32)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
    """))

    conn.execute(text("""
        INSERT INTO region (code, nom)
        SELECT
            reg_code,
            GROUP_CONCAT(DISTINCT `reg_nom` SEPARATOR ', ')
        FROM tmp_commune
        GROUP BY reg_code
    """))
    
    conn.commit()

In [16]:
# département
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE departement (
            code VARCHAR(10) PRIMARY KEY,
            nom VARCHAR(100)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
    """))

    conn.execute(text("""
        INSERT INTO departement
        SELECT
            dep_code,
            GROUP_CONCAT(DISTINCT `dep_nom` SEPARATOR ', ') as dep_nom
        FROM tmp_commune
        GROUP BY dep_code
    """))
    
    conn.commit()

In [17]:
# commune
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE commune (
            id_commune INT AUTO_INCREMENT PRIMARY KEY,
            code_insee VARCHAR(10),
            localisation INT NOT NULL,
            nom_standard VARCHAR(100),
            region SMALLINT(6),
            departement VARCHAR(10),
            population INT,
            superficie_hectare INT,
            densite DECIMAL(10,2),
            altitude_moyenne SMALLINT(6),
            altitude_minimale SMALLINT(6),
            altitude_maximale SMALLINT(6),
            CONSTRAINT fk_commune_localisation FOREIGN KEY (localisation)
                REFERENCES localisation (id_localisation),
            CONSTRAINT fk_commune_region FOREIGN KEY (region)
                REFERENCES region (id),
            CONSTRAINT fk_commune_departement FOREIGN KEY (departement)
                REFERENCES departement (code)
        ) ENGINE = InnoDB DEFAULT CHARSET = utf8mb4 COLLATE = utf8mb4_unicode_ci;
    """))
    
    conn.commit()

In [18]:
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO commune (
            code_insee,
            localisation,
            nom_standard,
            region,
            departement,
            population,
            superficie_hectare,
            densite,
            altitude_moyenne,
            altitude_minimale,
            altitude_maximale
        )
        SELECT
            code_insee,
            (select distinct id_localisation from localisation l where tc.longitude_mairie=l.longitude and tc.latitude_mairie=l.latitude),
            nom_standard,
            (select distinct id from region where tc.reg_code=code),
            dep_code,
            population,
            superficie_hectare,
            densite,
            altitude_moyenne,
            altitude_minimale,
            altitude_maximale
        FROM tmp_commune tc
    """))
    
    conn.commit()

In [19]:
# precision_surface
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE precision_surface (
            id SMALLINT(6) AUTO_INCREMENT PRIMARY KEY,
            nom varchar(32)
        ) ENGINE = InnoDB DEFAULT CHARSET = utf8mb4 COLLATE = utf8mb4_unicode_ci;
    """))

    conn.execute(text("""
        INSERT INTO precision_surface (nom)
        VALUES ('Non renseigné'),('Estimées'),('Mesurées')
    """))
    
    conn.commit()

In [20]:
# type_peuplement
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE type_peuplement (
            id SMALLINT(6) AUTO_INCREMENT PRIMARY KEY,
            nom varchar(32)
        ) ENGINE = InnoDB DEFAULT CHARSET = utf8mb4 COLLATE = utf8mb4_unicode_ci;
    """))

    conn.execute(text("""
        INSERT INTO type_peuplement (nom)
        VALUES('Landes / Garrigues / Maquis'),('Taillis'),('Futaies feuillues'),('Futaies résineuses'),('Futaies mélangées'),('Régénération / Reboisement'), ('Non renseigné')
    """)) 
    
    conn.commit()

In [21]:
# nature
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE nature (
            id SMALLINT(6) AUTO_INCREMENT PRIMARY KEY,
            nom varchar(32)
        ) ENGINE = InnoDB DEFAULT CHARSET = utf8mb4 COLLATE = utf8mb4_unicode_ci;
    """))

    conn.execute(text("""
        INSERT INTO nature (nom)
        VALUES('Naturelle'),('Involontaire (particulier)'),('Malveillance'),('Involontaire (travaux)'),('Accidentelle'),('Non renseigné')
    """))
    
    conn.commit()

In [22]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE incendie (
            id_incendie INT AUTO_INCREMENT PRIMARY KEY,
            localisation INT NOT NULL,
            code_insee VARCHAR(10) NOT NULL,
            date_premiere_alerte DATETIME NOT NULL,
            annee INT NOT NULL,
            surface_parcourue INT NOT NULL,
            surface_foret INT,
            surface_maquis_garrigues INT,
            autres_surfaces_naturelles INT,
            surfaces_agricoles INT,
            autres_surfaces INT,
            surface_autres_terres_boisees INT,
            surfaces_non_boisees_naturelles INT,
            surfaces_non_boisees_artificialisees INT,
            surfaces_non_boisees INT,
            precision_surface SMALLINT(6) NOT NULL,
            type_peuplement SMALLINT(6) NOT NULL,
            nature SMALLINT(6) NOT NULL,
            CONSTRAINT fk_incendie_localisation FOREIGN KEY (localisation)
                REFERENCES localisation (id_localisation),
            CONSTRAINT fk_incendie_precision_surface FOREIGN KEY (precision_surface)
                REFERENCES precision_surface (id),
            CONSTRAINT fk_incendie_type_peuplement FOREIGN KEY (type_peuplement)
                REFERENCES type_peuplement (id),
            CONSTRAINT fk_incendie_nature FOREIGN KEY (nature)
                REFERENCES nature (id)
        ) ENGINE = InnoDB DEFAULT CHARSET = utf8mb4 COLLATE = utf8mb4_unicode_ci;
    """))
    
    conn.commit()

In [25]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE tmp_incendie_no_insee AS
            SELECT * FROM tmp_incendie ti 
            LEFT JOIN tmp_commune tc ON ti.Insee = tc.code_insee
            WHERE tc.code_insee IS NULL
    """))
    
    conn.commit()

In [26]:
with engine.connect() as conn:
    conn.execute(text("""
        DELETE FROM tmp_incendie 
        WHERE
            Insee in
            (
                SELECT Insee FROM tmp_incendie_no_insee
            )
    """))
    
    conn.commit()

In [27]:
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO incendie (
            code_insee,
            localisation,
            date_premiere_alerte,
            annee,
            surface_parcourue,
            surface_foret,
            surface_maquis_garrigues,
            autres_surfaces_naturelles,
            surfaces_agricoles,
            autres_surfaces,
            surface_autres_terres_boisees,
            surfaces_non_boisees_naturelles,
            surfaces_non_boisees_artificialisees,
            surfaces_non_boisees,
            precision_surface,
            type_peuplement,
            nature
        )
        SELECT
            Insee,
            (select distinct id_localisation from localisation l, tmp_commune tc where tc.code_insee=insee and tc.longitude_mairie=l.longitude and tc.latitude_mairie=l.latitude),
            CAST(Date_Premiere_Alerte AS DATETIME),
            CAST(Annee AS SIGNED),
            Surface_Parcourue,
            Surface_Foret,
            Surface_Maquis_Garrigues,
            Autres_Surfaces_Naturelles,
            Surfaces_Agricoles,
            Autres_Surfaces,
            Surface_Autres_Terres_Boisees,
            Surfaces_Non_Boisees_Naturelles,
            Surfaces_Non_Boisees_Artificialisees,
            Surfaces_Non_Boisees,
            COALESCE(
                (SELECT id FROM precision_surface WHERE nom = Precision_Surfaces),
                (SELECT id FROM precision_surface WHERE nom = 'Non renseigné')
            ) AS precision_surface,
            COALESCE(
                (SELECT id FROM type_peuplement WHERE nom = Type_Peuplement),
                (SELECT id FROM type_peuplement WHERE nom = 'Non renseigné')
            ) AS type_peuplement,
            COALESCE(
                (SELECT id FROM nature WHERE nom = Nature),
                (SELECT id FROM nature WHERE nom = 'Non renseigné')
            ) AS nature
        FROM tmp_incendie
    """))

    conn.commit()

## Tables en lien avec cluster

In [28]:
# type_cluster
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE type_cluster (
            id SMALLINT AUTO_INCREMENT PRIMARY KEY,
            nom varchar(32)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
    """))

    conn.execute(text("""
        INSERT INTO type_cluster (nom)
        VALUES ('complete'),('with fire')
    """))
    
    conn.commit()

In [29]:
# cluster
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE cluster (
            id_localisation INT(11) NOT NULL,
            date_experiment DATETIME NOT NULL,
            cluster_id INT NOT NULL,
            type_cluster SMALLINT NOT NULL,
            PRIMARY KEY (id_localisation, date_experiment),
            CONSTRAINT fk_cluster_id_localisation FOREIGN KEY (id_localisation)
                REFERENCES localisation (id_localisation),
            CONSTRAINT fk_cluster_type_cluster FOREIGN KEY (type_cluster)
                REFERENCES type_cluster (id)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
    """))
    
    conn.commit()

In [30]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE affecte (
            id_commune INT NOT NULL,
            id_incendie INT(11) NOT NULL,
            date_impact DATETIME NOT NULL,
            degre_impact VARCHAR(50),
            PRIMARY KEY (id_commune, id_incendie, date_impact),
            CONSTRAINT fk_affecte_id_commune FOREIGN KEY (id_commune)
                REFERENCES commune (id_commune),
            CONSTRAINT fk_affecte_id_incendie FOREIGN KEY (id_incendie)
                REFERENCES incendie (id_incendie)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
    """))
    
    conn.commit()